<a href="https://colab.research.google.com/github/VintaBytes/Ciencia-de-datos/blob/main/libros/ciencia-de-datos-con-python/volumen-01/cuadernos/capitulo-019-fechas-y-datos-temporales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Capítulo 19 — Fechas y datos temporales

## Breve repaso

En el trabajo anterior sobre conversión de tipos de datos vimos que no alcanza con que una columna parezca numérica o parezca una fecha. Para que Pandas pueda trabajar correctamente con esos valores, necesita interpretarlos con el tipo adecuado.

Convertimos columnas como `Quantity`, `Price Per Unit` y `Total Spent` a formato numérico usando `pd.to_numeric()`. También hicimos una primera conversión de la columna `Transaction Date` usando `pd.to_datetime()`.

En este capítulo vamos a profundizar en el trabajo con fechas.

Las fechas son especialmente importantes porque permiten analizar los datos a lo largo del tiempo. En un dataset de ventas, por ejemplo, una columna de fecha puede ayudarnos a responder preguntas como: ¿en qué días hubo más transacciones?, ¿qué período cubre el dataset?, ¿hay ventas concentradas en ciertos días?, ¿cómo se distribuyen las operaciones a lo largo del tiempo?

Para responder este tipo de preguntas, no alcanza con que la fecha esté escrita como texto. Necesitamos que Pandas la interprete como dato temporal.

Una vez que una columna fue convertida a fecha, podemos ordenar registros cronológicamente, detectar valores fuera de rango, extraer el año, el mes o el día, y construir nuevas variables temporales útiles para el análisis.

También vamos a hacer un primer uso acotado de `groupby()`. Esta es una herramienta muy potente de Pandas para agrupar datos y calcular resúmenes por grupo. Más adelante la vamos a estudiar con mayor profundidad. En este capítulo la usaremos solamente para mostrar cómo una fecha bien convertida permite resumir información por períodos.

Al finalizar este capítulo deberías poder:

- Comprender por qué las fechas necesitan un tratamiento especial.
- Convertir una columna de texto a fecha usando `pd.to_datetime()`.
- Identificar valores temporales no interpretables.
- Revisar el rango temporal de un dataset.
- Ordenar datos usando una columna de fecha.
- Extraer componentes como año, mes, día y día de la semana.
- Crear columnas nuevas a partir de una fecha.
- Realizar primeras exploraciones temporales simples.

## Dataset de trabajo

Para trabajar con fechas vamos a volver al dataset real **Cafe Sales — Dirty Data for Cleaning Training**.

Este dataset contiene transacciones de venta de una cafetería. Una de sus columnas, `Transaction Date`, registra la fecha de cada operación.

Esa columna será el centro de este capítulo. Primero vamos a cargar el dataset, revisar cómo Pandas interpreta la fecha y luego convertirla a un tipo temporal adecuado.

Como cada capítulo debe poder ejecutarse de manera independiente, vamos a descargar y cargar nuevamente el dataset.

In [39]:
import kagglehub
import pandas as pd
import os

ruta_dataset = kagglehub.dataset_download(
    "ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training"
)

archivos = os.listdir(ruta_dataset)

archivos_csv = [
    archivo for archivo in archivos
    if archivo.endswith(".csv")
]

ruta_csv = os.path.join(ruta_dataset, archivos_csv[0])

df = pd.read_csv(ruta_csv)

df.head()

Using Colab cache for faster access to the 'cafe-sales-dirty-data-for-cleaning-training' dataset.


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


La salida de `head()` nos permite confirmar que el archivo fue cargado correctamente.

En este capítulo vamos a concentrarnos en la columna `Transaction Date`. Antes de transformarla, necesitamos revisar cómo fue cargada por Pandas.

## Revisar la columna de fecha

Antes de convertir una columna, conviene observar cómo fue cargada.

En este dataset, la columna que registra la fecha de cada transacción se llama `Transaction Date`.

Podemos revisar su tipo de dato:

In [40]:
df["Transaction Date"].dtype

dtype('O')

La salida indica que la columna fue cargada como `object`.

Eso significa que, por ahora, Pandas la está tratando como texto. Aunque los valores parezcan fechas, todavía no son fechas en sentido técnico para Pandas.

Podemos observar algunos valores:

In [41]:
df["Transaction Date"].head(10)

,Transaction Date
0,2023-09-08
1,2023-05-16
2,2023-07-19
3,2023-04-27
4,2023-06-11
5,2023-03-31
6,2023-10-06
7,2023-10-28
8,2023-07-28
9,2023-12-31


Visualmente, los valores pueden parecer fechas válidas. Pero mientras la columna esté como `object`, no podremos aprovechar correctamente las herramientas temporales de Pandas.

También podemos revisar cuántos valores faltantes tiene esta columna antes de convertirla:

In [42]:
df["Transaction Date"].isna().sum()

np.int64(159)

Este conteo nos da un primer diagnóstico.

Más adelante vamos a convertir la columna con `pd.to_datetime()` y compararemos si la cantidad de valores faltantes cambia. Si aumenta, eso indicará que había valores escritos como texto que no pudieron interpretarse como fechas.

## Convertir la columna a fecha

Ya vimos que `Transaction Date` fue cargada como `object`. Aunque sus valores parecen fechas, Pandas todavía la trata como texto.

Para convertirla a un tipo temporal usamos `pd.to_datetime()`.

Vamos a crear una copia del dataset para trabajar sin modificar directamente el `DataFrame` original.

In [43]:
df_fechas = df.copy()

df_fechas["Transaction Date"] = pd.to_datetime(
    df_fechas["Transaction Date"],
    errors="coerce"
)

df_fechas["Transaction Date"].head(10)

,Transaction Date
0,2023-09-08
1,2023-05-16
2,2023-07-19
3,2023-04-27
4,2023-06-11
5,2023-03-31
6,2023-10-06
7,2023-10-28
8,2023-07-28
9,2023-12-31


Ahora la columna fue convertida a formato de fecha.

Usamos `errors="coerce"` para que cualquier valor que no pueda interpretarse como fecha se convierta en `NaT`.

`NaT` significa *Not a Time*. Es el equivalente temporal de un valor faltante. Así como `NaN` representa un dato faltante en muchas columnas, `NaT` representa una fecha faltante o no interpretable.

Después de convertir, debemos verificar el tipo de dato resultante.

In [44]:
df_fechas["Transaction Date"].dtype

dtype('<M8[ns]')

El tipo `datetime64[ns]` indica que Pandas ya reconoce la columna como temporal.

Esto nos permite trabajar con herramientas específicas para fechas: ordenar cronológicamente, extraer año, mes o día, calcular rangos temporales y construir nuevas columnas derivadas de la fecha.

## Verificar valores no interpretables

Antes de convertir la columna `Transaction Date`, ya habíamos detectado 159 valores faltantes.

Después de usar `pd.to_datetime()` con `errors="coerce"`, conviene revisar si esa cantidad cambió.

Si la cantidad de faltantes aumenta, significa que algunos valores que no estaban vacíos no pudieron interpretarse como fechas y fueron convertidos en `NaT`.

In [45]:
faltantes_fecha_antes = df["Transaction Date"].isna().sum()
faltantes_fecha_despues = df_fechas["Transaction Date"].isna().sum()

print("Faltantes antes de convertir:")
print(faltantes_fecha_antes)

print()

print("Faltantes después de convertir:")
print(faltantes_fecha_despues)

Faltantes antes de convertir:
159

Faltantes después de convertir:
460


Si ambos valores coinciden, entonces la conversión no generó nuevos faltantes. Eso significa que los valores no faltantes de la columna pudieron interpretarse correctamente como fechas.

Si el segundo valor fuera mayor que el primero, deberíamos investigar qué valores no pudieron convertirse.

Esta comparación es importante porque `errors="coerce"` evita que la conversión se detenga con un error, pero también puede transformar valores problemáticos en `NaT`. Por eso, después de convertir fechas, siempre debemos verificar cuántos valores temporales faltantes quedaron.

El resultado muestra algo importante: después de convertir, la cantidad de valores faltantes aumentó.

Antes de la conversión había 159 valores faltantes en `Transaction Date`. Después de aplicar `pd.to_datetime()` con `errors="coerce"`, aparecen 460 valores temporales faltantes.

Eso significa que había valores no vacíos que Pandas no pudo interpretar como fechas. Al usar `errors="coerce"`, esos valores fueron convertidos en `NaT`.

Calculemos cuántos valores nuevos se volvieron faltantes durante la conversión.

In [46]:
nuevos_nat = faltantes_fecha_despues - faltantes_fecha_antes

nuevos_nat

np.int64(301)

El resultado indica cuántos valores no faltantes se convirtieron en `NaT`.

Estos casos merecen una revisión adicional. No eran valores vacíos, pero tampoco pudieron convertirse correctamente a fecha. Podrían ser textos como `"ERROR"`, `"UNKNOWN"` u otros valores problemáticos.

El siguiente paso es identificar cuáles eran esos valores originales.

## Revisar los valores que no pudieron convertirse

Ahora sabemos que la conversión generó nuevos valores `NaT`.

Para investigar esos casos, necesitamos comparar dos cosas:

```text
la columna original antes de convertir
la columna convertida a fecha
```

Nos interesan las filas donde la columna original no estaba vacía, pero la columna convertida quedó como `NaT`.

Podemos construir una condición para identificar esos casos.

In [47]:
fechas_no_convertidas = (
    df["Transaction Date"].notna()
    & df_fechas["Transaction Date"].isna()
)

fechas_no_convertidas.sum()

np.int64(301)

El resultado debería coincidir con la cantidad de nuevos valores `NaT`.

Ahora podemos observar qué valores originales tenían esas filas.

In [48]:
df.loc[fechas_no_convertidas, "Transaction Date"].value_counts(dropna=False)

,count
Transaction Date,
UNKNOWN,159
ERROR,142


Esta salida muestra cuáles eran los valores originales que no pudieron interpretarse como fechas.

Si aparecen valores como `"UNKNOWN"` o `"ERROR"`, eso confirma que la columna no tenía solamente fechas válidas y valores faltantes, sino también textos problemáticos.

Esta revisión es importante porque nos ayuda a diferenciar situaciones:

```text
NaN original        → la fecha ya estaba faltante antes de convertir
NaT por conversión  → había un valor escrito, pero no era una fecha interpretable
```

Ambos casos aparecen como faltantes después de la conversión, pero no significan exactamente lo mismo. Esa diferencia puede ser importante si queremos documentar la calidad del dataset.

## Revisar el rango temporal del dataset

Una vez que la columna fue convertida a fecha, podemos empezar a obtener información temporal útil.

Una primera pregunta importante es:

```text
¿Qué período cubre el dataset?
```

Para responderla podemos buscar la fecha mínima y la fecha máxima registradas.

In [49]:
fecha_minima = df_fechas["Transaction Date"].min()
fecha_maxima = df_fechas["Transaction Date"].max()

print("Fecha mínima:")
print(fecha_minima)

print()

print("Fecha máxima:")
print(fecha_maxima)

Fecha mínima:
2023-01-01 00:00:00

Fecha máxima:
2023-12-31 00:00:00


La fecha mínima indica el primer momento registrado en el dataset. La fecha máxima indica el último.

Esta revisión nos permite entender el rango temporal de los datos. Por ejemplo, podríamos descubrir que el dataset cubre varios meses, un año completo o solo un período breve.

También puede servir para detectar valores sospechosos. Si esperamos trabajar con ventas de un año determinado y aparece una fecha demasiado antigua o demasiado futura, esa fecha debería revisarse.

Cuando una columna está cargada como texto, este tipo de revisión puede ser menos confiable. Al convertirla a fecha, Pandas puede comparar los valores temporalmente y encontrar correctamente el mínimo y el máximo.

También podemos calcular la amplitud temporal del dataset, es decir, cuánto tiempo transcurre entre la primera y la última fecha registrada.

In [50]:
fecha_maxima - fecha_minima

Timedelta('364 days 00:00:00')

El resultado es una diferencia de tiempo.

Este valor nos da una idea general de la extensión temporal del dataset. A partir de esta información, más adelante podríamos decidir si tiene sentido analizar los datos por día, por mes, por trimestre o por otro período.

## Ordenar datos por fecha

Una vez que una columna fue convertida a tipo temporal, podemos usarla para ordenar el dataset cronológicamente.

Ordenar por fecha permite leer las transacciones en el orden en que ocurrieron. Esto puede ser útil para revisar la evolución de los datos, detectar períodos con mayor actividad o preparar análisis posteriores.

Vamos a ordenar el `DataFrame` según `Transaction Date`.

In [51]:
df_fechas.sort_values("Transaction Date").head(10)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
8015,TXN_4801947,Juice,1,3.0,3.0,Digital Wallet,Takeaway,2023-01-01
9063,TXN_9161256,Smoothie,2,4.0,8.0,Digital Wallet,In-store,2023-01-01
7309,TXN_6093955,Tea,5,1.5,NaN,UNKNOWN,Takeaway,2023-01-01
1425,TXN_8842223,Sandwich,5,NaN,20.0,Digital Wallet,In-store,2023-01-01
1777,TXN_7367474,Juice,5,3.0,15.0,Digital Wallet,Takeaway,2023-01-01
2557,TXN_2690222,ERROR,4,3.0,12.0,NaN,NaN,2023-01-01
768,TXN_5728991,Salad,ERROR,5.0,10.0,NaN,NaN,2023-01-01
1806,TXN_2192787,Sandwich,5,4.0,20.0,Cash,In-store,2023-01-01
1912,TXN_5563675,Tea,3,1.5,4.5,Digital Wallet,In-store,2023-01-01
9356,TXN_2104473,Cake,3,3.0,9.0,Digital Wallet,Takeaway,2023-01-01


La salida muestra las primeras transacciones según el orden temporal.

Como la columna ya fue convertida a fecha, Pandas no ordena los valores como simples textos, sino como fechas reales. Esto es importante porque el orden cronológico depende del significado temporal de los valores, no solamente de cómo están escritos.

También podemos mirar las últimas fechas del dataset:

In [52]:
df_fechas.sort_values("Transaction Date").tail(10)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
9838,TXN_6744733,Sandwich,1,4.0,NaN,Cash,Takeaway,NaT
9877,TXN_1964893,Smoothie,1,4.0,4.0,NaN,ERROR,NaT
9885,TXN_4659954,NaN,3,4.0,12.0,Credit Card,In-store,NaT
9907,TXN_8104914,Cookie,1,1.0,1.0,NaN,Takeaway,NaT
9931,TXN_8344810,Smoothie,2,4.0,8.0,NaN,UNKNOWN,NaT
9933,TXN_9460419,Cake,1,3.0,3.0,NaN,Takeaway,NaT
9937,TXN_8253472,Cake,1,3.0,3.0,NaN,NaN,NaT
9949,TXN_3130865,Juice,3,3.0,9.0,NaN,In-store,NaT
9983,TXN_9226047,Smoothie,3,4.0,12.0,Cash,NaN,NaT
9988,TXN_9594133,Cake,5,3.0,NaN,ERROR,NaN,NaT


Esta vista muestra las transacciones ubicadas al final del período registrado.

Al ordenar por fecha, es posible que los valores `NaT` aparezcan al final del resultado. Esto ocurre porque esos registros no tienen una fecha válida para ser ubicados cronológicamente.

Por eso, cuando trabajamos con fechas, debemos recordar que no todos los registros necesariamente podrán participar de un análisis temporal. Las filas con fechas faltantes o no interpretables requieren una decisión específica: conservarlas para otros análisis, excluirlas de ciertos cálculos temporales o revisarlas con más detalle.

## Extraer componentes de una fecha

Una vez que una columna está convertida a tipo fecha, podemos extraer partes específicas de cada valor temporal.

Por ejemplo, de una fecha podemos obtener:

```text
año
mes
día
día de la semana
```

Esto es útil porque muchas preguntas de análisis no se hacen sobre la fecha completa, sino sobre alguna de sus partes. Por ejemplo, podríamos querer saber si hay más transacciones en ciertos meses, si algunos días de la semana tienen más actividad o si el dataset cubre un solo año o varios.

En Pandas, para acceder a componentes de una fecha usamos `.dt`.

Primero vamos a crear una copia de trabajo con la columna de fecha ya convertida.

In [53]:
df_temporal = df_fechas.copy()

df_temporal["anio"] = df_temporal["Transaction Date"].dt.year
df_temporal["mes"] = df_temporal["Transaction Date"].dt.month
df_temporal["dia"] = df_temporal["Transaction Date"].dt.day

df_temporal[
    [
        "Transaction Date",
        "anio",
        "mes",
        "dia"
    ]
].head(10)

,Transaction Date,anio,mes,dia
0,2023-09-08,2023.0,9.0,8.0
1,2023-05-16,2023.0,5.0,16.0
2,2023-07-19,2023.0,7.0,19.0
3,2023-04-27,2023.0,4.0,27.0
4,2023-06-11,2023.0,6.0,11.0
5,2023-03-31,2023.0,3.0,31.0
6,2023-10-06,2023.0,10.0,6.0
7,2023-10-28,2023.0,10.0,28.0
8,2023-07-28,2023.0,7.0,28.0
9,2023-12-31,2023.0,12.0,31.0


Ahora el `DataFrame` tiene nuevas columnas derivadas de la fecha.

La columna `anio` contiene el año de la transacción. La columna `mes` contiene el número de mes. La columna `dia` contiene el día del mes.

Observemos que los valores pueden aparecer con `.0`, por ejemplo `2023.0` o `9.0`. Esto ocurre porque algunas filas tienen `NaT` en la fecha original. Al extraer año, mes o día de una fecha faltante, Pandas genera valores faltantes en las columnas derivadas, y por eso usa un formato numérico que puede representar esos faltantes.

Lo importante en este punto no es el formato visual, sino la idea: las columnas `anio`, `mes` y `dia` fueron construidas a partir de la fecha convertida.

Estas columnas no estaban en el dataset original. Las construimos a partir de `Transaction Date`.

Este tipo de transformación es muy útil porque permite preparar el dataset para preguntas temporales más específicas. En lugar de trabajar siempre con la fecha completa, podemos analizar los datos por año, mes o día.

## Extraer el día de la semana

Además del año, el mes y el día del mes, también podemos extraer el día de la semana.

Esto puede ser útil para analizar patrones de comportamiento. Por ejemplo, en un comercio podríamos preguntarnos si hay más transacciones los fines de semana, si ciertos productos se venden más en determinados días o si la actividad cambia entre días laborales y no laborales.

Pandas permite obtener el nombre del día usando `.dt.day_name()`.

In [54]:
df_temporal["dia_semana"] = df_temporal["Transaction Date"].dt.day_name()

df_temporal[
    [
        "Transaction Date",
        "dia_semana"
    ]
].head(10)

,Transaction Date,dia_semana
0,2023-09-08,Friday
1,2023-05-16,Tuesday
2,2023-07-19,Wednesday
3,2023-04-27,Thursday
4,2023-06-11,Sunday
5,2023-03-31,Friday
6,2023-10-06,Friday
7,2023-10-28,Saturday
8,2023-07-28,Friday
9,2023-12-31,Sunday


La columna `dia_semana` contiene el nombre del día correspondiente a cada fecha.

Por defecto, en este entorno los nombres de los días aparecen en inglés, por ejemplo `Friday`, `Tuesday` o `Wednesday`. Esto no afecta el análisis: siguen representando correctamente el día de la semana. Más adelante, si necesitáramos presentar los resultados en español, podríamos crear un reemplazo o mapeo de nombres.

Esta información no estaba escrita directamente en el dataset original. La obtuvimos a partir de la columna temporal.

In [55]:
df_temporal["numero_dia_semana"] = df_temporal["Transaction Date"].dt.dayofweek

df_temporal[
    [
        "Transaction Date",
        "dia_semana",
        "numero_dia_semana"
    ]
].head(10)

,Transaction Date,dia_semana,numero_dia_semana
0,2023-09-08,Friday,4.0
1,2023-05-16,Tuesday,1.0
2,2023-07-19,Wednesday,2.0
3,2023-04-27,Thursday,3.0
4,2023-06-11,Sunday,6.0
5,2023-03-31,Friday,4.0
6,2023-10-06,Friday,4.0
7,2023-10-28,Saturday,5.0
8,2023-07-28,Friday,4.0
9,2023-12-31,Sunday,6.0


En Pandas, `dayofweek` usa la siguiente convención:

```text
0 → lunes
1 → martes
2 → miércoles
3 → jueves
4 → viernes
5 → sábado
6 → domingo
```

El nombre del día resulta más fácil de leer, mientras que el número puede ser útil para ordenar los días correctamente o para construir algunos tipos de análisis.

En este capítulo vamos a usar estas columnas solo para primeras exploraciones temporales. Más adelante podremos profundizar en agrupamientos, resúmenes y visualizaciones por períodos.

## Primeras exploraciones temporales

Después de crear columnas como `anio`, `mes`, `dia` y `dia_semana`, podemos empezar a hacer preguntas simples sobre la distribución temporal de las transacciones.

Por ejemplo, podemos revisar cuántas transacciones hay por año.

In [56]:
df_temporal["anio"].value_counts(dropna=False)

,count
anio,
2023.0,9540
NaN,460


Este conteo nos permite ver si el dataset contiene registros de un solo año o de varios.

También podemos contar transacciones por mes:

In [57]:
df_temporal["mes"].value_counts(dropna=False).sort_index()

,count
mes,
1.0,818
2.0,727
3.0,827
4.0,774
5.0,777
6.0,818
7.0,791
8.0,803
9.0,788


En este caso usamos `sort_index()` para que los meses aparezcan ordenados numéricamente, del 1 al 12.

Este tipo de conteo todavía es simple, pero ya muestra el valor de haber convertido la columna de fecha. Ahora podemos analizar partes de la fecha que antes no estaban disponibles como columnas.

También podemos revisar las transacciones por día de la semana:

In [58]:
df_temporal["dia_semana"].value_counts(dropna=False)

,count
dia_semana,
Friday,1388
Monday,1382
Sunday,1380
Thursday,1380
Saturday,1358
Wednesday,1341
Tuesday,1311
NaN,460


Este conteo muestra cuántas transacciones corresponden a cada día de la semana.

Es posible que los días no aparezcan en orden calendario, porque `value_counts()` ordena por frecuencia de manera descendente. Más adelante podremos trabajar formas más ordenadas de presentar este tipo de resultados.

Por ahora, lo importante es notar que la columna de fecha nos permitió construir nuevas variables y empezar a explorar patrones temporales.

## Una primera agrupación temporal con groupby()

Hasta ahora usamos `value_counts()` para contar transacciones según año, mes o día de la semana.

Otra forma de resumir información por grupos es usar `groupby()`.

`groupby()` es una herramienta muy potente de Pandas. Permite agrupar filas según los valores de una o más columnas y luego calcular resúmenes para cada grupo. Más adelante vamos a estudiarla con mayor profundidad.

En este capítulo la vamos a usar solo de manera introductoria, para mostrar cómo una columna temporal puede servir para resumir información por períodos.

Por ejemplo, podemos contar cuántas transacciones hubo en cada fecha.

In [59]:
transacciones_por_fecha = (
    df_temporal
    .groupby("Transaction Date")["Transaction ID"]
    .count()
)

transacciones_por_fecha.head()

,Transaction ID
Transaction Date,
2023-01-01,20
2023-01-02,21
2023-01-03,21
2023-01-04,27
2023-01-05,38


En esta instrucción agrupamos el dataset por `Transaction Date` y contamos cuántos identificadores de transacción aparecen en cada fecha.

El resultado muestra la cantidad de transacciones registradas por día.

También podemos ordenar ese resultado para ver los días con mayor cantidad de transacciones:


In [60]:
transacciones_por_fecha.sort_values(ascending=False).head(10)

,Transaction ID
Transaction Date,
2023-02-06,40
2023-06-16,40
2023-09-21,39
2023-07-21,39
2023-07-24,39
2023-03-13,39
2023-01-05,38
2023-10-22,37
2023-01-25,37


Esta salida permite identificar las fechas con más transacciones registradas.

El uso de `groupby()` nos permite pasar de una tabla de transacciones individuales a un resumen por período. Esa es una de las razones por las que convertir correctamente las fechas es tan importante: una vez que Pandas entiende la columna como temporal, podemos construir análisis organizados por día, mes, año u otros períodos.

## Fechas faltantes o no interpretables

Cuando convertimos `Transaction Date`, vimos que no todas las filas quedaron con una fecha válida.

Antes de la conversión había valores faltantes reales. Después de convertir con `errors="coerce"`, algunos valores adicionales pasaron a ser `NaT` porque no pudieron interpretarse como fechas.

Esto es importante porque las filas con `NaT` no pueden ubicarse correctamente en una línea de tiempo. No pueden asignarse a un año, a un mes, a un día ni a un día de la semana.

Podemos revisar cuántas fechas no interpretables o faltantes quedaron en `df_temporal`.

In [61]:
df_temporal["Transaction Date"].isna().sum()

np.int64(460)

Ese conteo nos indica cuántas filas no tienen una fecha válida después de la conversión.

También podemos observar algunas de esas filas.

In [62]:
df_temporal[df_temporal["Transaction Date"].isna()].head(10)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,anio,mes,dia,dia_semana,numero_dia_semana
11,TXN_3051279,Sandwich,2,4.0,8.0,Credit Card,Takeaway,NaT,NaN,NaN,NaN,NaN,NaN
29,TXN_7640952,Cake,4,3.0,12.0,Digital Wallet,Takeaway,NaT,NaN,NaN,NaN,NaN,NaN
33,TXN_7710508,UNKNOWN,5,1.0,5.0,Cash,NaN,NaT,NaN,NaN,NaN,NaN,NaN
77,TXN_2091733,Salad,1,5.0,5.0,NaN,In-store,NaT,NaN,NaN,NaN,NaN,NaN
103,TXN_7028009,Cake,4,3.0,12.0,NaN,Takeaway,NaT,NaN,NaN,NaN,NaN,NaN
104,TXN_7447872,Juice,2,NaN,6.0,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN
115,TXN_1001832,Salad,2,5.0,10.0,Cash,Takeaway,NaT,NaN,NaN,NaN,NaN,NaN
142,TXN_7943008,Coffee,1,2.0,2.0,Credit Card,NaN,NaT,NaN,NaN,NaN,NaN,NaN
160,TXN_1093800,Sandwich,3,4.0,12.0,Cash,Takeaway,NaT,NaN,NaN,NaN,NaN,NaN
175,TXN_6463132,Cookie,5,1.0,5.0,Credit Card,Takeaway,NaT,NaN,NaN,NaN,NaN,NaN


Estas filas no deberían desaparecer automáticamente del dataset. Pueden seguir siendo útiles para otros análisis, como productos vendidos, métodos de pago o importes, siempre que esas columnas tengan datos válidos.

Sin embargo, para análisis temporales específicos, estas filas requieren una decisión. Podemos excluirlas de ciertos conteos por fecha, conservarlas como registros sin fecha, revisar si el valor original era `"UNKNOWN"` o `"ERROR"`, o intentar recuperar la fecha desde otra fuente si existiera.

La decisión depende del contexto y del objetivo del análisis.

En este capítulo no vamos a corregir esas fechas. Lo importante es reconocer que una conversión temporal puede dejar filas fuera del análisis por período, y que eso debe ser registrado.

## Errores frecuentes al trabajar con fechas

Al trabajar con fechas, uno de los errores más frecuentes es asumir que una columna es temporal solo porque sus valores se ven como fechas.

Una fecha escrita como `"2023-09-08"` puede parecer correcta para una persona, pero si Pandas la cargó como `object`, todavía es texto. En ese estado, la columna no permite aprovechar correctamente las herramientas temporales de Pandas.

Por eso, antes de analizar fechas, conviene revisar el tipo de dato:

```python
df["Transaction Date"].dtype
```

Otro error frecuente es convertir fechas sin verificar cuántos valores quedaron como `NaT`. Cuando usamos `pd.to_datetime()` con `errors="coerce"`, los valores que no pueden interpretarse como fechas se convierten en `NaT`. Esto evita que el código se detenga, pero también puede ocultar problemas si no revisamos el resultado.

Por eso, después de convertir, conviene comparar faltantes antes y después:

```python
df["Transaction Date"].isna().sum()
df_fechas["Transaction Date"].isna().sum()
```

También puede ser un error eliminar automáticamente las filas con fechas faltantes o no interpretables. Esas filas no sirven para ciertos análisis temporales, pero pueden seguir siendo útiles para otros análisis. Por ejemplo, una venta sin fecha válida todavía podría aportar información sobre producto, cantidad, importe o método de pago.

Otro punto importante es no abrir demasiadas conclusiones solo a partir de conteos temporales simples. Saber que un día tuvo más transacciones que otro puede ser una señal interesante, pero antes de sacar conclusiones deberíamos considerar el contexto: cantidad de días registrados, posibles datos faltantes, períodos incompletos o sesgos de carga.

Finalmente, cuando extraemos columnas como `anio`, `mes`, `dia` o `dia_semana`, debemos recordar que esas columnas dependen de que la fecha original esté correctamente convertida. Si una fila tiene `NaT`, las columnas derivadas también quedarán sin información válida.

Una buena rutina para trabajar con fechas podría ser:

```text
revisar el tipo de dato original
convertir la columna con pd.to_datetime()
comparar faltantes antes y después
revisar valores no interpretables
analizar el rango temporal
crear columnas temporales derivadas
verificar los resultados antes de sacar conclusiones
```

Trabajar con fechas no consiste solamente en cambiar el tipo de una columna. Implica asegurarse de que los valores temporales sean interpretables, revisar qué registros quedan fuera del análisis temporal y usar las nuevas variables con cuidado.

## Resumen del capítulo

En este capítulo trabajamos con fechas y datos temporales.

Partimos de una idea central: una fecha escrita como texto no es lo mismo que una fecha interpretada como dato temporal por Pandas. Aunque valores como `"2023-09-08"` parezcan fechas válidas a simple vista, si la columna está cargada como `object`, Pandas todavía la trata como texto.

Primero revisamos la columna `Transaction Date` y comprobamos que había sido cargada como `object`:

```python
df["Transaction Date"].dtype
```

También observamos algunos valores y contamos los faltantes antes de la conversión. En ese momento detectamos 159 valores faltantes reales en la columna original.

Luego convertimos la columna usando `pd.to_datetime()`:

```python
df_fechas["Transaction Date"] = pd.to_datetime(
    df_fechas["Transaction Date"],
    errors="coerce"
)
```

El parámetro `errors="coerce"` permitió convertir en `NaT` los valores que no podían interpretarse como fechas. Después de la conversión, la cantidad de valores faltantes temporales aumentó a 460. Esto indicó que, además de los faltantes originales, había valores escritos que no podían interpretarse como fechas.

Para investigar esos casos, construimos una condición que identificó las filas donde la columna original no estaba vacía, pero la columna convertida quedó como `NaT`:

```python
fechas_no_convertidas = (
    df["Transaction Date"].notna()
    & df_fechas["Transaction Date"].isna()
)
```

Ese paso nos permitió diferenciar entre valores que ya estaban faltantes antes de convertir y valores que se volvieron `NaT` durante la conversión.

Después revisamos el rango temporal del dataset usando la fecha mínima y la fecha máxima. Esto nos permitió saber qué período cubren los datos y también pensar en posibles valores fuera de rango.

Más adelante ordenamos el dataset por fecha y vimos que, una vez convertida la columna, Pandas puede ordenar cronológicamente los registros.

Luego creamos nuevas columnas temporales a partir de `Transaction Date`:

```python
df_temporal["anio"] = df_temporal["Transaction Date"].dt.year
df_temporal["mes"] = df_temporal["Transaction Date"].dt.month
df_temporal["dia"] = df_temporal["Transaction Date"].dt.day
```

También extraímos el día de la semana:

```python
df_temporal["dia_semana"] = df_temporal["Transaction Date"].dt.day_name()
df_temporal["numero_dia_semana"] = df_temporal["Transaction Date"].dt.dayofweek
```

Estas nuevas columnas permiten hacer preguntas temporales más específicas, como cuántas transacciones hubo por año, por mes o por día de la semana.

Finalmente hicimos una primera agrupación temporal con `groupby()`. Usamos esta herramienta de manera introductoria para contar transacciones por fecha:

```python
transacciones_por_fecha = (
    df_temporal
    .groupby("Transaction Date")["Transaction ID"]
    .count()
)
```

Más adelante vamos a estudiar `groupby()` con mayor profundidad. En este capítulo lo usamos solo para mostrar que una columna temporal bien convertida permite resumir información por períodos.

La idea principal de este capítulo fue:

```text
Trabajar con fechas requiere convertirlas, verificarlas y luego usarlas para construir nuevas variables temporales.
```

Las fechas permiten analizar evolución, períodos, patrones y distribución temporal. Pero antes de hacerlo, necesitamos asegurarnos de que Pandas las interprete correctamente y de que sepamos qué registros quedaron sin fecha válida.

## Próximo paso

Ya trabajamos con valores faltantes, duplicados, limpieza de textos, conversión de tipos y fechas.

El siguiente paso será comenzar a integrar varias de estas tareas en un proceso de limpieza más completo. En lugar de tratar cada problema por separado, vamos a trabajar sobre un flujo de preparación de datos paso a paso.

La idea será partir de un dataset con varios problemas, aplicar transformaciones ordenadas y verificar el resultado final. Esto nos acercará a una situación más realista de trabajo: preparar una tabla para que quede lista para analizar.